In [1]:
%pip install google-cloud-bigquery pyarrow db-dtypes

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pmdarima 2.1.1 requires statsmodels>=0.14.5, but you have statsmodels 0.14.2 which is incompatible.
tensorflow-intel 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 7.36.2 which is incompatible.


  Obtaining dependency information for google-cloud-bigquery from https://files.pythonhosted.org/packages/4b/48/79129b85526710828e84e83cb66efad3680587ac63dfc30ae6eb7ab2fb7b/google_cloud_bigquery-3.47.0-py3-none-any.whl.metadata
  Obtaining dependency information for db-dtypes from https://files.pythonhosted.org/packages/a8/d6/c0e6434ff96161687e6d7b535b124a9135a6cefbae5aaa1a37402777e9b9/db_dtypes-1.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for google-api-core[grpc]<3.0.0,>=2.28.0 from https://files.pythonhosted.org/packages/c9/73/4068d4b123d206f42e44d476b6ed4de773d81efc4e88101ff82e318b9a7a/google_api_core-2.44.0-py3-none-any.whl.metadata
  Obtaining dependency information for google-cloud-core<3.0.0,>=2.4.1 from https://files.pythonhosted.org/packages/12/ed/1b09640a565e5d34d4e517463f65a67a11f9dcdeb6b6e8220f2d06aaf489/google_cloud_core-2.8.0-py3-none-any.whl.metadata
  Obtaining dependency information for google-resumable-media<3.0.0,>=2.0.0 from https://files.pyth

In [5]:
# Import library Panda
import pandas as pd
import os
import matplotlib.pyplot as plt

### Main trx and Forecasting data dari local

In [7]:
main_trx = pd.read_csv('D:\Portofolio Data Analyst and Scientist\Regional-Product-Trends-Through-Time-Series-Forecasting\Implementation\kaggle_upload\main_trx.csv')
print(main_trx.shape)

(3095137, 8)


In [10]:
main_trx['transaction_date'] = pd.to_datetime(main_trx['transaction_date'])

In [11]:
main_trx

,id,user_id,transaction_date,product_name,nama_provinsi,nama_kabupaten,nama_kecamatan,total_amount
0,TRX501069,SC1312102002037,2024-12-31,Woman Fashion,SUMATERA BARAT,PASAMAN BARAT,LUHAK NAN DUO,173400.0
1,TRX668723,SC3522202002006,2024-12-31,Sewa Motor/Mobil,JAWA TIMUR,BOJONEGORO,KASIMAN,0.0
2,TRX1041898,SC3506192003014,2024-12-31,Otomotif,JAWA TIMUR,KEDIRI,KANDANGAN,10200.0
3,TRX2620150,SC3523192010003,2024-12-31,"Tagihan (WIFI, PLN)",JAWA TIMUR,TUBAN,WIDANG,100.0
4,TRX1650440,SC3507272005013,2024-12-31,Food & Drink,JAWA TIMUR,MALANG,NGANTANG,7400.0
...,...,...,...,...,...,...,...,...
3095132,TRX1818640,SC3516062013017,2024-01-01,Food & Drink,JAWA TIMUR,MOJOKERTO,PUNGGING,35900.0
3095133,TRX830367,SC1274051003007,2024-01-01,"Transportasi (Kereta, Pesawat, Kapal)",SUMATERA UTARA,KOTA TANJUNG BALAI,DATUK BANDAR,8100.0
3095134,TRX2278627,SC3513142010012,2024-01-01,Sewa Motor/Mobil,JAWA TIMUR,PROBOLINGGO,KRAKSAAN,5500.0
3095135,TRX864115,SC1223031001007,2024-01-01,Man Fashion,SUMATERA UTARA,LABUHANBATU UTARA,KUALUH HILIR,24100.0


In [13]:
from google.cloud import bigquery

PROJECT_ID = 'scsqlfoundation'
DATASET = 'maintrx'

client = bigquery.Client(project=PROJECT_ID)

# Hapus tabel tes lama (aman, tidak error kalau belum ada)
client.delete_table(f'{PROJECT_ID}.{DATASET}.tes_koneksi', not_found_ok=True)

job = client.load_table_from_dataframe(
    main_trx.head(1000),
    f'{PROJECT_ID}.{DATASET}.tes_koneksi'
)
job.result()
print('Berhasil')

c:\Users\Rebecca YBS\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Berhasil


In [14]:
from google.cloud import bigquery

PROJECT_ID = 'scsqlfoundation'
DATASET = 'maintrx'

client = bigquery.Client(project=PROJECT_ID)

# Pastikan kolom tanggal bertipe datetime
main_trx['transaction_date'] = pd.to_datetime(main_trx['transaction_date'])

table_id = f'{PROJECT_ID}.{DATASET}.main_trx'

# Hapus tabel lama kalau ada, supaya skema tidak bentrok
client.delete_table(table_id, not_found_ok=True)

chunk_size = 500_000

for i in range(0, len(main_trx), chunk_size):
    chunk = main_trx.iloc[i:i+chunk_size]
    job = client.load_table_from_dataframe(
        chunk,
        table_id,
        job_config=bigquery.LoadJobConfig(
            write_disposition='WRITE_TRUNCATE' if i == 0 else 'WRITE_APPEND'
        )
    )
    job.result()
    print(f'{min(i+chunk_size, len(main_trx))} / {len(main_trx)} baris')

c:\Users\Rebecca YBS\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


500000 / 3095137 baris


c:\Users\Rebecca YBS\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


1000000 / 3095137 baris


c:\Users\Rebecca YBS\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


1500000 / 3095137 baris


... (4 outputs left out)

c:\Users\Rebecca YBS\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


3000000 / 3095137 baris


c:\Users\Rebecca YBS\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


3095137 / 3095137 baris


In [15]:
import pandas as pd
from pathlib import Path

folder = Path(r'D:\Portofolio Data Analyst and Scientist\Regional-Product-Trends-Through-Time-Series-Forecasting\Implementation\kaggle_upload_forecast')

dfs = {}

for file in sorted(folder.iterdir()):
    if file.suffix == '.csv':
        dfs[file.stem] = pd.read_csv(file)

print(f'{len(dfs)} file terbaca')
for name, df in dfs.items():
    print(name, df.shape)

12 file terbaca
jatim_fd_pred (8, 3)
jatim_mf_pred (8, 3)
jatim_rh_pred (8, 3)
maluku_krt_pred (8, 3)
maluku_otomotif_pred (8, 3)
maluku_wf_pred (8, 3)
sumbar_e_pred (8, 3)
sumbar_krt_pred (8, 3)
sumbar_wf_pred (8, 3)
sumut_fd_pred (8, 3)
sumut_rh_pred (8, 3)
sumut_wf_pred (8, 3)


In [16]:
for name, df in dfs.items():
    if 'ds' in df.columns:
        df['ds'] = pd.to_datetime(df['ds'])

In [18]:
from google.cloud import bigquery

PROJECT_ID = 'scsqlfoundation'
DATASET_FC = 'maintrx'

client = bigquery.Client(project=PROJECT_ID)

for name, df in dfs.items():
    table_id = f'{PROJECT_ID}.{DATASET_FC}.{name}'
    client.delete_table(table_id, not_found_ok=True)
    job = client.load_table_from_dataframe(
        df,
        table_id,
        job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')
    )
    job.result()
    print(f'{name}: {len(df)} baris')

c:\Users\Rebecca YBS\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


jatim_fd_pred: 8 baris


c:\Users\Rebecca YBS\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


jatim_mf_pred: 8 baris


c:\Users\Rebecca YBS\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


jatim_rh_pred: 8 baris


... (14 outputs left out)

c:\Users\Rebecca YBS\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


sumut_rh_pred: 8 baris


c:\Users\Rebecca YBS\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


sumut_wf_pred: 8 baris
